In [14]:
import torch
import torch.nn as nn

import numpy as np

from src.gtsrb import NUM_CLASSES, GTSRB_CLASSES, get_dataloaders, save_predictions

In [2]:
train_loader, val_loader, test_loader = get_dataloaders(img_size=32, batch_size=128)

In [3]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

c:\Users\elisa\Documents\Visão computacional\aaavenv\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


torch.Size([128, 3, 32, 32])
torch.Size([128])


In [5]:
model = nn.Sequential(

    nn.Conv2d(3, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Flatten(),

    nn.Linear(64 * 8 * 8, 128),
    nn.ReLU(),

    nn.Linear(128, NUM_CLASSES)
)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

Sequential(
  (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=4096, out_features=128, bias=True)
  (8): ReLU()
  (9): Linear(in_features=128, out_features=43, bias=True)
)

In [7]:
criterion = nn.CrossEntropyLoss()

In [8]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [9]:
epochs = 5

for epoch in range(epochs):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}")


    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()

            total += labels.size(0)

    accuracy = 100 * correct / total

    print(f"Validation Accuracy: {accuracy:.2f}%")

Epoch 1, Loss: 294.2478
Validation Accuracy: 79.02%
Epoch 2, Loss: 61.1953
Validation Accuracy: 92.83%
Epoch 3, Loss: 26.7533
Validation Accuracy: 96.47%
Epoch 4, Loss: 14.4010
Validation Accuracy: 97.82%
Epoch 5, Loss: 9.6720
Validation Accuracy: 97.86%


In [ ]:
model.eval()

all_preds = []

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        all_preds.append(predictions.cpu())

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

    accuracy = 100 * correct / total

    print(f"Test Global Accuracy: {accuracy:.2f}%")
    #print(acc)


y_pred = torch.cat(all_preds)



c:\Users\elisa\Documents\Visão computacional\aaavenv\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


tensor(0.)
(tensor(0), 1)
tensor(8.)
(tensor(1), 1)
tensor(4.)
(tensor(1), 1)
tensor(7.)
(tensor(1), 1)
tensor(5.)
(tensor(1), 1)
tensor(4.)
(tensor(1), 1)
tensor(0.)
(tensor(0), 1)
tensor(2.)
(tensor(1), 1)
tensor(2.)
(tensor(1), 1)
tensor(7.)
(tensor(1), 1)
tensor(9.)
(tensor(1), 1)
tensor(5.)
(tensor(1), 1)
tensor(6.)
(tensor(1), 1)
tensor(5.)
(tensor(1), 1)
tensor(2.)
(tensor(1), 1)
tensor(2.)
(tensor(1), 1)
tensor(4.)
(tensor(1), 1)
tensor(2.)
(tensor(1), 1)
tensor(2.)
(tensor(1), 1)
tensor(0.)
(tensor(0), 1)
tensor(1.)
(tensor(1), 1)
tensor(2.)
(tensor(1), 1)
tensor(1.)
(tensor(1), 1)
tensor(0.)
(tensor(1), 1)
tensor(0.)
(tensor(1), 1)
tensor(6.)
(tensor(1), 1)
tensor(2.)
(tensor(1), 1)
tensor(2.)
(tensor(1), 1)
tensor(1.)
(tensor(1), 1)
tensor(0.)
(tensor(1), 1)
tensor(3.)
(tensor(1), 1)
tensor(0.)
(tensor(0), 1)
tensor(1.)
(tensor(1), 1)
tensor(6.)
(tensor(1), 1)
tensor(1.)
(tensor(1), 1)
tensor(4.)
(tensor(1), 1)
tensor(0.)
(tensor(0), 1)
tensor(0.)
(tensor(0), 1)
tensor(5.)
(

In [ ]:
model.eval()

cm = np.zeros((43,43), dtype=int)

with torch.no_grad():
    for data in test_loader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = model(images)
        _, predictions = torch.max(outputs, 1)

        for label, prediction in zip(labels, predictions):
            cm[label, prediction] += 1

print(cm)

[[ 26  25   0 ...   0   0   0]
 [  0 695   5 ...   0   0   0]
 [  2  25 697 ...   0   0   0]
 ...
 [  0   0   0 ...  31   0   0]
 [  0   0   0 ...   0  30   3]
 [  0   0   0 ...   0   2  82]]


In [13]:
acc_class = [0 for c in GTSRB_CLASSES]
for i in range(43):
  acc = 100 * (float(cm[i,i]) / np.sum(cm[i, :]))
  print(f'Acurácia {i}: {acc:.2f}')
  acc_class[i] = acc

print(f"Worst class: {acc_class.index(min(acc_class))} - {min(acc_class):.2f}%")
print(f"Best class: {acc_class.index(max(acc_class))} - {max(acc_class):.2f}%")
print(f"")

Acurácia 0: 43.33
Acurácia 1: 96.53
Acurácia 2: 92.93
Acurácia 3: 80.89
Acurácia 4: 91.36
Acurácia 5: 88.57
Acurácia 6: 77.33
Acurácia 7: 64.89
Acurácia 8: 88.00
Acurácia 9: 90.21
Acurácia 10: 92.88
Acurácia 11: 88.10
Acurácia 12: 97.68
Acurácia 13: 99.44
Acurácia 14: 90.74
Acurácia 15: 96.67
Acurácia 16: 95.33
Acurácia 17: 90.28
Acurácia 18: 78.21
Acurácia 19: 95.00
Acurácia 20: 86.67
Acurácia 21: 51.11
Acurácia 22: 89.17
Acurácia 23: 60.67
Acurácia 24: 77.78
Acurácia 25: 88.33
Acurácia 26: 75.56
Acurácia 27: 41.67
Acurácia 28: 88.67
Acurácia 29: 83.33
Acurácia 30: 30.67
Acurácia 31: 85.19
Acurácia 32: 98.33
Acurácia 33: 100.00
Acurácia 34: 96.67
Acurácia 35: 86.67
Acurácia 36: 95.00
Acurácia 37: 48.33
Acurácia 38: 91.30
Acurácia 39: 36.67
Acurácia 40: 34.44
Acurácia 41: 50.00
Acurácia 42: 91.11
Worst class: 30 - 30.67%
Best class: 33 - 100.00%



In [15]:
save_predictions(
    y_pred,
    "results/predicoes_baseline.csv",
    experiment_name="CNN Baseline"
)